In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_excel("/Users/rhugvedbhojane/Desktop/Coursework/Fall 2025/LH Pamli Project /data/raw/IV.xlsx")
df.head(10)

,Date,SPY Close,ATM Vol 1m,95% Moneyness Vol 1m,105% Moneyness Vol 1m
0,2023-01-03,380.82,22.0717,24.3927,19.5396
1,2023-01-04,383.76,21.2113,23.3036,18.7624
2,2023-01-05,379.38,21.7447,23.6395,19.2657
3,2023-01-06,388.08,20.4760,22.5052,17.6294
4,2023-01-09,387.86,20.6416,23.2387,18.4528
5,2023-01-10,390.58,19.6920,21.9412,17.2102
6,2023-01-11,395.52,20.1339,22.5684,17.8215
7,2023-01-12,396.96,17.7333,20.3711,15.8236
8,2023-01-13,398.50,16.9458,20.2739,14.7321
9,2023-01-17,397.77,17.9244,21.0877,15.5214


In [3]:
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date")

In [4]:
df["ret"] = np.log(df["SPY Close"] / df["SPY Close"].shift(1))

In [5]:
window = 21
trading_days = 252
df["realized_21d"] = (
    df["ret"].rolling(window).std() * np.sqrt(trading_days)
)

In [6]:
df["ATM_IV_1m"] = df["ATM Vol 1m"] / 100.0
df["Put95_IV_1m"] = df["95% Moneyness Vol 1m"] / 100.0
df["Call105_IV_1m"] = df["105% Moneyness Vol 1m"] / 100.0

In [7]:
df["vrp"] = df["ATM_IV_1m"] - df["realized_21d"]

In [8]:
df["skew_rr"] = df["Put95_IV_1m"] - df["Call105_IV_1m"]

In [9]:
df_clean = df.dropna(subset=["realized_21d", "vrp", "skew_rr"])

In [10]:
trade_date = pd.to_datetime("2025-04-09")
current_row = df_clean[df_clean["Date"] <= trade_date].sort_values("Date").tail(1)

In [11]:
stats = df_clean[["realized_21d", "ATM_IV_1m", "vrp", "skew_rr"]].describe()
print(stats)

       realized_21d   ATM_IV_1m         vrp     skew_rr
count    554.000000  554.000000  554.000000  554.000000
mean       0.133462    0.140732    0.007270    0.066768
std        0.055040    0.039541    0.039290    0.014897
min        0.057587    0.093356   -0.249186    0.025280
25%        0.103242    0.113405   -0.007281    0.054889
50%        0.123062    0.129466    0.009329    0.065623
75%        0.146526    0.156124    0.029466    0.075721
max        0.506940    0.423233    0.111191    0.158511


In [12]:
curr_skew = current_row["skew_rr"].iloc[0]
curr_vrp = current_row["vrp"].iloc[0]

In [13]:
skew_percentile = (df_clean["skew_rr"] < curr_skew).mean() * 100
vrp_percentile = (df_clean["vrp"] < curr_vrp).mean() * 100

In [14]:
print(f"\nCurrent skew percentile: {skew_percentile:.1f}%")
print(f"Current VRP percentile:  {vrp_percentile:.1f}%")


Current skew percentile: 99.8%
Current VRP percentile:  0.5%
